# Metadata filters and permissions

RAG retrieval must respect the same authorization boundary as the source system. This notebook filters documents before building a retriever and proves that another tenant's evidence is never returned.

## Security boundary

```mermaid
flowchart LR
 U[User identity] --> P[Authorization filter]
 D[Document index] --> P --> R[Retriever]
 R --> C[Context]
 P -.-> X[Unauthorized docs excluded]
```

Do not rely on the model to hide text it was given. Filtering after retrieval is too late.

In [ ]:
from examples.intermediate.permission_filter import SecureDocument, User, authorized_documents, secure_search

documents = [
    SecureDocument('acme-api', 'Acme API key rotation procedure', 'acme', frozenset({'support'})),
    SecureDocument('globex-api', 'Globex API key rotation procedure', 'globex', frozenset({'support'})),
    SecureDocument('acme-internal', 'Acme payroll policy', 'acme', frozenset({'hr'})),
]
user = User('u-1', 'acme', frozenset({'support'}))
[doc.doc_id for doc in authorized_documents(user, documents)]

In [ ]:
results = secure_search(user, 'API key rotation', documents)
[(doc.doc_id, round(score, 2)) for doc, score in results]
assert all(doc.tenant_id == 'acme' for doc, _ in results)

## Exercise

Add an expiration timestamp or role requirement. Test same-tenant authorized access, same-tenant denied tags, and cross-tenant exact matches. Then explain how you would apply the filter in Qdrant, OpenSearch, or another production store.